Interface Implementation: The Palestinian Liberation Navigator
==============================================================

1\. Architectural Overview & Framework
--------------------------------------

The user interface is built using **Gradio**, an open-source Python library specifically designed for rapid deployment of machine learning models. Gradio was selected for its native compatibility with Google Colab and its ability to handle complex transformers inference logic within a simplified, responsive web container.

### Dependency Management

To ensure an "end-to-end" execution experience as required by the rubric, the script includes an install\_dependencies() function. This dynamically checks for and installs the necessary libraries (gradio, peft, bitsandbytes, etc.) at runtime, preventing environment-related crashes for the end user.

2\. Hardware Adaptability & Quantization
----------------------------------------

A critical feature of this interface is its **Hardware-Aware Loading Logic**. The script detects whether the environment has an NVIDIA GPU (cuda) or is restricted to a cpu.

*   **GPU Path (4-bit Quantization):** If a GPU is detected, the model is loaded using BitsAndBytesConfig in 4-bit (NF4). This shrinks the 2.5B parameter Gemma model to roughly 2.5GB of VRAM, allowing for near-instant inference.
    
*   **CPU Path (Full Precision Fallback):** If no GPU is found, the model falls back to standard 32-bit floating-point math. While slower, this ensures the assistant remains accessible to users without specialized hardware, adhering to the principle of "Democratic Computing."
    
*   **Adapter Injection:** The script uses the peft library to "snap on" the trained LoRA adapters from the ./gemma-revolutionary-assistant directory onto the base Gemma model.
    

3\. Inference Logic & Structural Framing
----------------------------------------

The assistant’s reasoning is governed by a dedicated generate\_response function that enforces the ideological and technical boundaries of the project.

### I. Prompt Engineering

The function constructs a strict prompt template:

> (### Instruction:\\n{user\_query}\\n\\n### Response:\\n.)

This matches the format used during the fine-tuning phase, ensuring the model remains in its "Revolutionary Navigator" persona.

### II. The "StopOnInstruction" Mechanism

Small models often suffer from "hallucination," where they continue generating text beyond the answer (often inventing new questions). I implemented a custom StoppingCriteria class that detects the string (###)Instruction in the output stream and immediately cuts the generation. This ensures the responses remain concise and grounded.

### III. Parameter Control

*   **Repetition Penalty (1.15):** Applied to prevent the "looping" behavior common in 2B parameter models.
    
*   **Temperature (0.6):** A balanced setting that allows for creative, structural analysis without sacrificing factual consistency.
    

4\. User Interface Design & Aesthetics
--------------------------------------

To satisfy the "Exemplary" criteria for UI integration, the interface was designed with a specific "Constructivist" aesthetic that mirrors the project's de-colonial mission.

*   **Iconography:** A centered Palestinian flag serves as the primary visual anchor, immediately establishing the domain context.
    
*   **Theming:** I utilized the Monochrome theme with a primary\_hue="red". This high-contrast red/gray/white palette evokes the look of a digital underground press.


    
*   **Instructional Context:** A bolded CSS-styled paragraph explains the "counter-hegemonic" nature of the model, providing the user with immediate clarity on how to interact with the assistant.
    
*   **Curated Examples:** Three "Golden Queries" are provided as clickable buttons. These examples (ICJ Case, BDS/PUMA, and Legal Rebuttals) guide the user toward the model’s areas of maximum expertise.
    

5\. Summary of User Experience
------------------------------

The final interface is fully responsive and provides clear feedback (via a gr.Warning for CPU users). It transforms a complex neural network into an intuitive tool where users can input queries and receive structural, de-colonial analysis without needing to understand the underlying PyTorch code.

In [ ]:
import os
import torch
import subprocess
import sys

def install_dependencies():
    """Ensures all required libraries are installed for the UI."""
    print("Checking UI dependencies...")
    packages = ["gradio", "transformers", "peft", "bitsandbytes", "accelerate"]
    for package in packages:
        try:
            __import__(package)
        except ImportError:
            print(f"Installing {package}...")
            subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", package])

install_dependencies()

import gradio as gr
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig, StoppingCriteria, StoppingCriteriaList
from peft import PeftModel

# ==========================================
# 0. HARDWARE CHECK & AUTHENTICATION
# ==========================================
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"--- Hardware Detected: {DEVICE.upper()} ---")
if DEVICE == "cpu":
    print("Notice: Running on CPU. Responses will generate slower (~1-3 words/sec).")

# Fetch Hugging Face Token for the Base Gemma Model
try:
    from google.colab import userdata
    HF_TOKEN = userdata.get('HF_TOKEN')
except Exception:
    HF_TOKEN = os.environ.get('HF_TOKEN')

if not HF_TOKEN:
    print("WARNING: HF_TOKEN not found. The gated Gemma model may fail to load.")

class StopOnInstruction(StoppingCriteria):
    """Prevents the model from hallucinating a second question."""
    def __init__(self, target_ids):
        self.target_ids = target_ids

    def __call__(self, input_ids: torch.LongTensor, scores: torch.FloatTensor, **kwargs) -> bool:
        for target in self.target_ids:
            if input_ids[0][-len(target):].tolist() == target:
                return True
        return False

# ==========================================
# 1. CONFIGURATION & LOADING
# ==========================================
BASE_MODEL_ID = "google/gemma-2b-it"
ADAPTER_PATH = "./gemma-revolutionary-assistant" # Folder containing your downloaded LoRA files

print(f"Loading Tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL_ID, token=HF_TOKEN)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

print(f"Loading Base Model ({BASE_MODEL_ID})...")
if DEVICE == "cuda":
    # GPU Loading (Fast & Memory Efficient)
    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.float16,
    )
    base_model = AutoModelForCausalLM.from_pretrained(
        BASE_MODEL_ID,
        quantization_config=bnb_config,
        device_map="auto",
        token=HF_TOKEN
    )
else:
    # CPU Loading (Requires standard 32-bit floats and high system RAM)
    base_model = AutoModelForCausalLM.from_pretrained(
        BASE_MODEL_ID,
        device_map="cpu",
        torch_dtype=torch.float32,
        token=HF_TOKEN,
        low_cpu_mem_usage=True
    )

try:
    print(f"Applying de-colonial LoRA adapters from {ADAPTER_PATH}...")
    model = PeftModel.from_pretrained(base_model, ADAPTER_PATH)
    model.eval()
    print("Success: Revolutionary Assistant is armed and ready.")
except Exception as e:
    print(f"\n[!] ERROR LOADING ADAPTER: {e}")
    print("Falling back to the basic, pre-trained corporate model for testing.")
    model = base_model

# Prepare stopping criteria
stop_token_ids = [tokenizer.encode("\n### Instruction", add_special_tokens=False)]
stopping_criteria = StoppingCriteriaList([StopOnInstruction(stop_token_ids)])

# ==========================================
# 2. INFERENCE LOGIC
# ==========================================
def generate_response(instruction, temperature, max_tokens):
    # Construct prompt exactly as it was during fine-tuning
    prompt = f"### Instruction:\n{instruction}\n\n### Response:\n"

    inputs = tokenizer(prompt, return_tensors="pt").to(DEVICE)
    input_length = inputs.input_ids.shape[1]

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_tokens,
            temperature=temperature,
            top_p=0.9,
            repetition_penalty=1.15,      # Helps prevent looping phrases
            do_sample=True if temperature > 0.1 else False,
            pad_token_id=tokenizer.eos_token_id,
            stopping_criteria=stopping_criteria
        )

    # Extract only the newly generated tokens
    new_tokens = outputs[0][input_length:]
    response = tokenizer.decode(new_tokens, skip_special_tokens=True).strip()

    # Final cleanup
    if "### Instruction" in response:
        response = response.split("### Instruction")[0].strip()

    return response

# ==========================================
# 3. GRADIO UI DESIGN
# ==========================================
# We use a custom theme to give it a more structural, organized feel
custom_theme = gr.themes.Monochrome(
    primary_hue="red",
    secondary_hue="gray",
    font=[gr.themes.GoogleFont("Inter"), "ui-sans-serif", "system-ui", "sans-serif"]
)

with gr.Blocks(title="The Liberatory Assistant", theme=custom_theme) as demo:

    # Header Design - Centered around a prominent image
    gr.HTML("""
    <div style="text-align: center; max-width: 900px; margin: 0 auto; padding: 20px;">
        <img src="https://upload.wikimedia.org/wikipedia/commons/thumb/0/00/Flag_of_Palestine.svg/1200px-Flag_of_Palestine.svg.png"
             style="max-height: 180px; width: auto; margin: 0 auto 25px auto; border-radius: 8px; box-shadow: 0 6px 12px rgba(0,0,0,0.15);"
             alt="Revolutionary Banner">

        <h1 style="color: #b30000; font-family: Impact, sans-serif; letter-spacing: 2px; margin-bottom: 5px;">
            THE PALESTINIAN LIBERATION NAVIGATOR
        </h1>
        <h3 style="color: #333; margin-top: 0;">
            De-Colonial Solidarity, Legal Resistance & BDS Assistant
        </h3>
        <p style="font-size: 1.1em; line-height: 1.5; font-style: italic; color: #555; background-color: #f9f9f9; padding: 15px; border-left: 4px solid #b30000; border-radius: 4px;">
            "This LLM is fine-tuned on counter-hegemonic datasets. It prioritizes the material reality of the Palestinian struggle,
            international legal frameworks (such as the ICJ), and global solidarity campaigns like the Boycott, Divestment,
            and Sanctions (BDS) movement over standard corporate narratives."
        </p>
    </div>
    """)

    if DEVICE == "cpu":
        gr.Warning("You are running on a CPU. Responses will take a bit longer to generate. Grab a coffee!")

    with gr.Row():
        with gr.Column(scale=2):
            input_text = gr.Textbox(
                label="Submit your query for structural analysis:",
                placeholder="e.g., Explain the legal basis for South Africa's case at the ICJ.",
                lines=4
            )
            submit_btn = gr.Button("Analyze Structurally", variant="primary", size="lg")

        with gr.Column(scale=1):
            temp = gr.Slider(0.1, 1.0, value=0.6, step=0.1, label="Dialectic Creativity (Temperature)")
            tokens = gr.Slider(50, 500, value=250, step=50, label="Max Response Length")

    output_text = gr.Textbox(label="Assistant Response", lines=8)

    # Updated Examples derived directly from the 'mlibre/palestine' dataset scope
    gr.Examples(
        examples=[
            ["Explain the legal basis for South Africa's genocide case against Israel at the ICJ under the Genocide Convention."],
            ["Why does the BDS movement call for a boycott of PUMA and how does it relate to illegal settlements?"],
            ["Provide a rebuttal to the claim: 'South Africa did not follow proper legal procedures in filing its case at the ICJ.'"]
        ],
        inputs=input_text
    )

    submit_btn.click(fn=generate_response, inputs=[input_text, temp, tokens], outputs=output_text)

if __name__ == "__main__":
    # share=True creates a public link you can share
    demo.launch(share=True, debug=True)

Checking UI dependencies...
--- Hardware Detected: CPU ---
Notice: Running on CPU. Responses will generate slower (~1-3 words/sec).
Loading Tokenizer...


`torch_dtype` is deprecated! Use `dtype` instead!


Loading Base Model (google/gemma-2b-it)...


Loading weights:   0%|          | 0/164 [00:00<?, ?it/s]

Applying de-colonial LoRA adapters from ./gemma-revolutionary-assistant...


Success: Revolutionary Assistant is armed and ready.


/tmp/ipython-input-556333751.py:143: DeprecationWarning: The 'theme' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'theme' to Blocks.launch() instead.
  with gr.Blocks(title="The Liberatory Assistant", theme=custom_theme) as demo:
/usr/local/lib/python3.12/dist-packages/gradio/helpers.py:1141: UserWarning: You are running on a CPU. Responses will take a bit longer to generate. Grab a coffee!
  warnings.warn(message)


Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://13f5b687606c415172.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [1]:
!unzip /content/gemma-revolutionary-model.zip

Archive:  /content/gemma-revolutionary-model.zip
   creating: gemma-revolutionary-assistant/
   creating: gemma-revolutionary-assistant/checkpoint-146/
  inflating: gemma-revolutionary-assistant/checkpoint-146/training_args.bin  
  inflating: gemma-revolutionary-assistant/checkpoint-146/trainer_state.json  
  inflating: gemma-revolutionary-assistant/checkpoint-146/tokenizer_config.json  
  inflating: gemma-revolutionary-assistant/checkpoint-146/tokenizer.json  
  inflating: gemma-revolutionary-assistant/checkpoint-146/scheduler.pt  
  inflating: gemma-revolutionary-assistant/checkpoint-146/rng_state.pth  
  inflating: gemma-revolutionary-assistant/checkpoint-146/optimizer.pt  
  inflating: gemma-revolutionary-assistant/checkpoint-146/chat_template.jinja  
  inflating: gemma-revolutionary-assistant/checkpoint-146/adapter_model.safetensors  
  inflating: gemma-revolutionary-assistant/checkpoint-146/adapter_config.json  
  inflating: gemma-revolutionary-assistant/checkpoint-146/README.md 